In [ ]:
import torch
import torch.nn as nn
import timm
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b4', pretrained=True)
        num_features = self.backbone.classifier.in_features

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.backbone(x)

print("Initializing model...")
model = DeepfakeDetector().to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Initializing model...
Total parameters: 18,467,658


In [ ]:
import os
print(os.path.exists('/content/genvscan_final-3.pth'))

True


In [ ]:
model.load_state_dict(torch.load('/content/genvscan_weights_only.pth', map_location=device, weights_only=True))
model.eval()

DeepfakeDetector(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
          (bn1): BatchNormAct2d(
            48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv

In [ ]:
CONFIG = {
    'batch_size': 32,
    'num_epochs': 20,
    'learning_rate': 1e-4,
    'image_size': 224,
    'num_workers': 2
}

val_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
def get_tta_transforms():
    base_norm = {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]}
    return [
        transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize(**base_norm)]),
        transforms.Compose([transforms.Resize((224, 224)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(**base_norm)]),
        transforms.Compose([transforms.Resize((256, 256)), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(**base_norm)]),
        transforms.Compose([transforms.Resize((256, 256)), transforms.CenterCrop(224), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(**base_norm)]),
        transforms.Compose([transforms.Resize((288, 288)), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(**base_norm)]),
    ]

tta = get_tta_transforms()

print("Upload a video to analyze:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
duration = total_frames / fps
num_frames = min(max(int(duration * 2), 16), 64)
frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

all_probs = []
all_confs = []

for idx in frame_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret:
        continue
    frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frame_probs = []
    for t in tta:
        tensor = t(frame_pil).unsqueeze(0).to(device)
        with torch.no_grad():
            p = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
        frame_probs.append(p)
    probs = np.mean(frame_probs, axis=0)
    all_probs.append(probs)
    all_confs.append(max(probs))

cap.release()

all_probs = np.array(all_probs)
all_confs = np.array(all_confs)
weights = all_confs / all_confs.sum()
weighted_probs = np.average(all_probs, axis=0, weights=weights)
final_label = "AI-Generated" if np.argmax(weighted_probs) == 1 else "Real"

result = {
    'label': final_label,
    'confidence': max(weighted_probs) * 100,
    'real': weighted_probs[0] * 100,
    'fake': weighted_probs[1] * 100,
    'frames': len(all_probs),
    'duration': duration
}
result

Upload a video to analyze:


In [ ]:
print(f'''
Detection: {result['label']}
Confidence: {result['confidence']:.2f}%
Real: {result['real']:.2f}%
AI-Generated: {result['fake']:.2f}%
Frames: {result['frames']}
Duration: {result['duration']:.1f}s
''')

In [ ]:
! pip install groq
from groq import Groq

client = Groq(api_key="-")

system_prompt = f"""You are Spectra, an AI-powered video authenticity detection assistant.

Your purpose: Help users understand whether a video is real or AI-generated.

Capabilities:
- You analyze videos using a deep learning detection model (EfficientNet-B4)
- You explain detection results in simple terms
- You point out visual cues that indicate real or AI-generated content

Personality:
- Concise and direct
- Friendly but professional
- Always respond in the same language the user uses (English or Arabic)

Current Detection Results:
- Prediction: {result['label']}
- Confidence: {result['confidence']:.2f}%

Only share the prediction and confidence with the user.
"""

messages = [{"role": "system", "content": system_prompt}]

while True:
    user_input = input("You: ")
    if user_input.lower() in ['quit', 'exit']:
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    print(f"\nSpectra: {reply}\n")